### Algoritmo Cliente

In [49]:
# librerias

import random
import numpy as np
from collections import defaultdict
from typing import List, Dict, Set, Tuple
import copy

In [50]:
# Clase cálculo de métricas de la distribución

class MetricasDistribucion:
    def __init__(self, asignaciones, asignacion_estudiantes, estudiantes, desafios):
        self.asignaciones = asignaciones
        self.asignacion_estudiantes = asignacion_estudiantes
        self.estudiantes = estudiantes
        self.desafios = desafios

    def calcular_satisfacccion_promedio(self):
        """
        Calcula la satisfacción promedio de la distribución según (1 / preferencia_de_la_asignacion + 1) por cada estudiante
        """
        satisfaccion_total = 0
        cantidad_estudiantes = len(self.estudiantes)

        for estudiante in self.estudiantes:
            desafio_asignado = self.asignacion_estudiantes.get(estudiante['Nombre'])
            if desafio_asignado in estudiante['Postulaciones']:
                preferencia_index = estudiante['Postulaciones'].index(desafio_asignado)
                satisfaccion = 1 / (preferencia_index + 1)
            else:
                satisfaccion = 0
            satisfaccion_total += satisfaccion

        return satisfaccion_total / cantidad_estudiantes

    def cantidad_primeras_preferencias(self):
        """
        Cuenta cuántos estudiantes quedaron en su primera preferencia
        """
        estudiantes_en_prioridad_1 = 0

        for estudiante in self.estudiantes:
            if (len(estudiante['Postulaciones']) > 0 and self.asignacion_estudiantes.get(estudiante['Nombre']) == estudiante['Postulaciones'][0]):
                estudiantes_en_prioridad_1 += 1

        return estudiantes_en_prioridad_1

    def cantidad_ninguna_preferencia(self):
        """
        Cuenta cuántos estudiantes quedaron fuera de sus preferencias
        """
        estudiantes_fuera_de_preferencias = 0

        for estudiante in self.estudiantes:
            desafio_asignado = self.asignacion_estudiantes.get(estudiante['Nombre'])
            if desafio_asignado not in estudiante['Postulaciones']:
                estudiantes_fuera_de_preferencias += 1

        return estudiantes_fuera_de_preferencias

    def cantidad_desafios_sin_equipo_valido(self):
        """
        Cuenta cuántos desafíos quedaron sin equipo (menos de 2 estudiantes)
        """
        desafios_sin_equipo = 0
        desafios_con_equipo = set(self.asignaciones.keys())

        # Contar desafíos no vacío pero no válido
        for desafio in desafios_con_equipo:
            if len(self.asignaciones[desafio]) < 2:
                desafios_sin_equipo += 1

        # Agregar desafíos con equipo vacío
        desafios_n = {desafio['Titulo'] for desafio in self.desafios}
        desafios_sin_equipo += len(desafios_n - desafios_con_equipo)

        return desafios_sin_equipo

    def calcular_equipos_STD(self):
        """
        Calcula la desviación estándar del tamaño de los equipos que tienen estudiantes
        """
        tamaño_equipos = [len(equipo) for equipo in self.asignaciones.values() if len(equipo) > 0]
        return np.std(tamaño_equipos) if tamaño_equipos else 0

    def calcular_media_de_carreras_en_equipo(self):
        """
        Calcula el promedio de carreras diferentes por equipo
        """
        carreras_por_equipo = []

        for equipo in self.asignaciones.values():
            if len(equipo) >= 2:  # Solo considerar equipos válidos
                carreras_unicas = len(set(estudiante['Carrera'] for estudiante in equipo))
                carreras_por_equipo.append(carreras_unicas)

        return np.mean(carreras_por_equipo) if carreras_por_equipo else 0

    def calcular_metricas(self):
        """
        Calcula todas las métricas
        """
        metricas = {
            'satisfaccion_promedio': self.calcular_satisfacccion_promedio(),
            'estudiantes_primera_prioridad': self.cantidad_primeras_preferencias(),
            'estudiantes_fuera_preferencias': self.cantidad_ninguna_preferencia(),
            'desafios_sin_equipo': self.cantidad_desafios_sin_equipo_valido(),
            'std_tamaño_equipos': self.calcular_equipos_STD(),
            'promedio_carreras_por_equipo': self.calcular_media_de_carreras_en_equipo()
        }

        return metricas

    def entregar_metricas(self):
        """
        Imprime todas las métricas
        """
        metricas = self.calcular_metricas()

        print("\n=== Métricas de Asignación ===")
        print(f"Satisfacción promedio: {metricas['satisfaccion_promedio']:.3f}")
        print(f"Estudiantes en primera prioridad: {metricas['estudiantes_primera_prioridad']}")
        print(f"Estudiantes fuera de preferencias: {metricas['estudiantes_fuera_preferencias']}")
        print(f"Desafíos sin equipo: {metricas['desafios_sin_equipo']}")
        print(f"Desviación estándar tamaño equipos: {metricas['std_tamaño_equipos']:.3f}")
        print(f"Promedio de carreras por equipo: {metricas['promedio_carreras_por_equipo']:.2f}")

        # Estadísticas adicionales
        total_estudiantes = len(self.estudiantes)
        print(f"\nPorcentajes:")
        print(f"Primera prioridad: {(metricas['estudiantes_primera_prioridad']/total_estudiantes)*100:.1f}%")
        print(f"Segunda y Tercera prioridad: {100 - (metricas['estudiantes_primera_prioridad']/total_estudiantes)*100 - (metricas['estudiantes_fuera_preferencias']/total_estudiantes)*100:.1f}%")
        print(f"Fuera de preferencias: {(metricas['estudiantes_fuera_preferencias']/total_estudiantes)*100:.1f}%")

In [51]:
# Clase: Distribuidor de estudiantes a desafíos

class GenerarEquipos:
    def __init__(self, data):
        self.estudiantes = data['estudiantes']
        self.desafios = data['desafios']
        self.carreras = data['carreras']
        self.asignaciones = {}
        self.asignacion_estudiantes = {}
        self.estudiantes_no_asignados = set(estudiante['Nombre'] for estudiante in self.estudiantes)
        self.estudiantes_dict = {estudiante['Nombre']: estudiante for estudiante in self.estudiantes}

        # Mantener preferencias originales intactas
        self.preferencias_originales = {
            estudiante['Nombre']: estudiante['Postulaciones'].copy() for estudiante in self.estudiantes
        }
        # Crear copia de trabajo de las preferencias
        self.preferencias_actuales = {
            estudiante['Nombre']: estudiante['Postulaciones'].copy() 
            for estudiante in self.estudiantes
        }

        # entregar los equipos tal cual se formaron (mayor a menor cantidad de postulaciones)
        self.orden_asignacion = []

    # 
    def postulaciones_por_desafio(self) -> Dict[str, int]:
        """Cuenta las postulaciones de prioridad 1 por desafío."""
        cantidad_postulaciones = defaultdict(int)
        for estudiante in self.estudiantes_no_asignados:
            if self.preferencias_actuales[estudiante]:  # Si tiene postulaciones pendientes
                # Solo contar la primera prioridad actual
                cantidad_postulaciones[self.preferencias_actuales[estudiante][0]] += 1
        return cantidad_postulaciones
    
    def limite_por_carrera(self, carrera: str) -> int:
        """Obtiene el límite máximo de estudiantes por carrera."""
        if carrera == "Ingeniería Civil Telemática":
            return 3
        return 1

    def verificar_asignacion_valida(self, estudiante: dict, desafio: str) -> bool:
        """Verifica si un estudiante puede ser añadido a un desafío."""

        # Verificar si el estudiante ya está asignado
        if estudiante['Nombre'] in self.asignacion_estudiantes:
            return False
            
        if desafio not in self.asignaciones:
            return True
            
        equipo_actual = self.asignaciones[desafio]
        
        # Verificar límite de equipo - Estricto máximo de 4
        if len(equipo_actual) >= 4:
            return False
            
        # Contar estudiantes por carrera en el equipo actual
        cantidad_carreras_en_desafio = defaultdict(int)
        for miembro in equipo_actual:
            cantidad_carreras_en_desafio[miembro['Carrera']] += 1
            
        # Verificar límite de carrera
        limite_carrera_estudiante = self.limite_por_carrera(estudiante['Carrera'])
        if cantidad_carreras_en_desafio[estudiante['Carrera']] >= limite_carrera_estudiante:
            return False
            
        return True

    def actualizar_prioridades_no_asignados(self, desafio: str):
        """Actualiza las prioridades de los estudiantes no asignados."""
        for estudiante in list(self.estudiantes_no_asignados):
            if desafio in self.preferencias_actuales[estudiante]:
                self.preferencias_actuales[estudiante].remove(desafio)

    def asignar_estudiante_a_desafio(self, estudiante: dict, desafio: str) -> bool:
        """Asigna un estudiante a un desafío y actualiza los registros."""
        if estudiante['Nombre'] in self.asignacion_estudiantes:
            return False

        if desafio not in self.asignaciones:
            self.asignaciones[desafio] = []
            self.orden_asignacion.append(desafio)
            
        if len(self.asignaciones[desafio]) >= 4:
            return False
            
        self.asignaciones[desafio].append(estudiante)
        self.asignacion_estudiantes[estudiante['Nombre']] = desafio
        self.estudiantes_no_asignados.discard(estudiante['Nombre'])
        return True

    def obtener_preferencia_original(self, estudiante: str, desafio: str) -> str:
        """Obtiene el número de preferencia original del estudiante para un desafío."""
        preferencia_original = self.preferencias_originales[estudiante]
        if desafio in preferencia_original:
            return f"Preferencia #{preferencia_original.index(desafio) + 1}"
        return "Fuera de preferencias"

    def obtener_posibles_estudiantes(self, desafio: str, estudiantes: List[dict]) -> List[dict]:
        """
        Obtiene una lista de estudiantes compatibles para un desafío,
        considerando las restricciones de carrera.
        """
        estudiantes_compatibles = []
        cuantos_por_carrera = defaultdict(int)
        
        for estudiante in estudiantes:
            # Verificar si agregar este estudiante excedería el límite de su carrera
            if cuantos_por_carrera[estudiante['Carrera']] >= self.limite_por_carrera(estudiante['Carrera']):
                continue
                
            # Si llegamos aquí, el estudiante es compatible
            estudiantes_compatibles.append(estudiante)
            cuantos_por_carrera[estudiante['Carrera']] += 1
            
            # Si ya tenemos 4 estudiantes compatibles, es suficiente
            if len(estudiantes_compatibles) >= 4:
                break
        
        return estudiantes_compatibles

    def entregar_resultados_asignacion(self):
        """Imprime las estadísticas de los equipos en orden de asignación."""
        print("\nEstadísticas de asignación:")
        print("==========================")
        
        for desafio in self.orden_asignacion:
            equipo = self.asignaciones[desafio]
                
            print(f"\nDesafío: {desafio}")
            print(f"Número de estudiantes: {len(equipo)}")
            
            cuantos_por_carrera = defaultdict(int)
            for estudiante in equipo:
                cuantos_por_carrera[estudiante['Carrera']] += 1
            
            print("Distribución por carrera:")
            for carrera, cantidad in cuantos_por_carrera.items():
                max_permitido = self.limite_por_carrera(carrera)
                print(f"- {carrera}: {cantidad} (máximo permitido: {max_permitido})")
            
            print("Estudiantes en el equipo:")
            for estudiante in equipo:
                preferencia = self.obtener_preferencia_original(estudiante['Nombre'], desafio)
                print(f"- {estudiante['Nombre']} ({estudiante['Carrera']}) ({preferencia})")

    def validar_asignacion(self) -> bool:
        """Valida que todas las asignaciones cumplan las restricciones."""
        total_estudiantes_asignados = len(self.asignacion_estudiantes)
        if total_estudiantes_asignados != len(self.estudiantes):
            print(f"Error: No todos los estudiantes están asignados. Asignados: {total_estudiantes_asignados}, Total: {len(self.students)}")
            return False

        for desafio, equipo in self.asignaciones.items():
            # Verificar límites de tamaño de equipo
            if len(equipo) < 2:  # Mínimo absoluto de 2 estudiantes
                print(f"Error: Equipo con menos de 2 estudiantes en {desafio}")
                return False
            if len(equipo) > 4:  # Máximo de 4 estudiantes
                print(f"Error: Equipo con más de 4 estudiantes en {desafio}, {len(equipo)}")
                return False
                
            # Verificar límites por carrera
            cuantos_por_carrera = defaultdict(int)
            for estudiante in equipo:
                cuantos_por_carrera[estudiante['Carrera']] += 1
                if cuantos_por_carrera[estudiante['Carrera']] > self.limite_por_carrera(estudiante['Carrera']):
                    print(f"Error: Demasiados estudiantes de {estudiante['Carrera']} en {desafio}")
                    return False
        
        # Imprimir advertencia sobre equipos pequeños (no es un error)
        equipos_pequeños = [desafio for desafio, equipo in self.asignaciones.items() if len(equipo) < 3]
        if equipos_pequeños:
            print("\nAdvertencia: Los siguientes desafíos tienen equipos de 2 integrantes:")
            for desafio in equipos_pequeños:
                print(f"- {desafio}")
        
        return True

    def asignar_estudiantes_restantes(self):
        """
        Asigna los estudiantes restantes asegurando que siempre se formen equipos
        de al menos 2 estudiantes y máximo 4.
        """
        while len(self.estudiantes_no_asignados) >= 2:
            estudiantes_restantes = [self.estudiantes_dict[nombre] for nombre in self.estudiantes_no_asignados]
            subgrupo_encontrado = False
            
            # Primero intentar completar equipos existentes pequeños
            for desafio, equipo in self.asignaciones.items():
                if len(equipo) < 3:  # Priorizar equipos pequeños
                    cuantos_por_carrera = defaultdict(int)
                    for miembro in equipo:
                        cuantos_por_carrera[miembro['Carrera']] += 1
                    
                    compatible_para_equipo = []
                    for estudiante in estudiantes_restantes:
                        if (cuantos_por_carrera[estudiante['Carrera']] < self.limite_por_carrera(estudiante['Carrera']) and self.verificar_asignacion_valida(estudiante, desafio)):
                            compatible_para_equipo.append(estudiante)
                            
                    if len(compatible_para_equipo) >= 2:
                        # Intentar agregar dos estudiantes compatibles
                        estudiantes_para_agregar = compatible_para_equipo[:2]
                        for estudiante in estudiantes_para_agregar:
                            if self.verificar_asignacion_valida(estudiante, desafio):
                                self.asignar_estudiante_a_desafio(estudiante, desafio)
                        subgrupo_encontrado = True
                        break
            
            if not subgrupo_encontrado:
                # Intentar crear un nuevo equipo
                for desafio in [c['Titulo'] for c in self.desafios]:
                    if desafio not in self.asignaciones:
                        cuantos_por_carrera = defaultdict(int)
                        estudiantes_compatibles = []
                        
                        for estudiante in estudiantes_restantes:
                            if cuantos_por_carrera[estudiante['Carrera']] < self.limite_por_carrera(estudiante['Carrera']):
                                estudiantes_compatibles.append(estudiante)
                                cuantos_por_carrera[estudiante['Carrera']] += 1
                                if len(estudiantes_compatibles) >= 2:
                                    # Crear nuevo equipo con los estudiantes compatibles
                                    for estudiante_compatible in estudiantes_compatibles[:2]:
                                        self.asignar_estudiante_a_desafio(estudiante_compatible, desafio)
                                    subgrupo_encontrado = True
                                    break
                        if subgrupo_encontrado:
                            break
            
            if not subgrupo_encontrado:
                break

        # Manejar estudiantes individuales restantes
        for estudiante_nombre in list(self.estudiantes_no_asignados):
            estudiante = self.estudiantes_dict[estudiante_nombre]
            for desafio, equipo in self.asignaciones.items():
                if len(equipo) >= 2 and len(equipo) < 4:
                    cuantos_por_carrera = defaultdict(int)
                    for miembro in equipo:
                        cuantos_por_carrera[miembro['Carrera']] += 1
                    
                    if (cuantos_por_carrera[estudiante['Carrera']] < self.limite_por_carrera(estudiante['Carrera']) and self.verificar_asignacion_valida(estudiante, desafio)):
                        self.asignar_estudiante_a_desafio(estudiante, desafio)
                        break

    def completar_equipos(self):
        """Completa equipos asegurando mínimo 3 estudiantes cuando sea posible y respetando máximo 4."""
        equipos_incompletos = [desafio for desafio, equipo in self.asignaciones.items() if len(equipo) < 3]
        
        for desafio in equipos_incompletos:
            equipo_actual = self.asignaciones[desafio]
            
            # Intentar mover estudiantes de equipos grandes
            for otro_desafio, equipo_grande in self.asignaciones.items():
                if otro_desafio == desafio or len(equipo_actual) >= 4:
                    continue
                if len(equipo_grande) > 3:  # Solo tomar de equipos que pueden ceder estudiantes
                    for estudiante in equipo_grande[:]:
                        if len(equipo_actual) < 3 and self.verificar_asignacion_valida(estudiante, desafio):
                            equipo_grande.remove(estudiante)
                            del self.asignacion_estudiantes[estudiante['Nombre']]
                            self.asignar_estudiante_a_desafio(estudiante, desafio)
                            if len(equipo_actual) >= 3:
                                break

    def formar_equipos(self) -> Tuple[Dict[str, List[dict]], Dict[str, str]]:
        """Formar los equipos."""
        while self.estudiantes_no_asignados:
            postulaciones_desafio = self.postulaciones_por_desafio()
            if not postulaciones_desafio:
                break
                
            desafio_actual = max(postulaciones_desafio.items(), key=lambda x: x[1])[0]
            
            estudiantes_prioridad_1 = [self.estudiantes_dict[estudiante_nombre] for estudiante_nombre in self.estudiantes_no_asignados if estudiante_nombre in self.estudiantes_dict and self.preferencias_actuales[estudiante_nombre] and self.preferencias_actuales[estudiante_nombre][0] == desafio_actual]
            
            if len(estudiantes_prioridad_1) >= 2:
                random.shuffle(estudiantes_prioridad_1)
                
                # Mantener conteo de carreras para este equipo
                cuantos_por_carrera = defaultdict(int)
                miembros_en_equipo = []
                
                # Primero encontrar estudiantes compatibles respetando límites de carrera
                for estudiante in estudiantes_prioridad_1:
                    if (cuantos_por_carrera[estudiante['Carrera']] < self.limite_por_carrera(estudiante['Carrera']) and len(miembros_en_equipo) < 4):
                        miembros_en_equipo.append(estudiante)
                        cuantos_por_carrera[estudiante['Carrera']] += 1
                
                # Solo formar equipo si tenemos al menos 2 estudiantes compatibles
                if len(miembros_en_equipo) >= 2:
                    for estudiante in miembros_en_equipo:
                        self.asignar_estudiante_a_desafio(estudiante, desafio_actual)
            
            # Actualizar prioridades de los no asignados
            self.actualizar_prioridades_no_asignados(desafio_actual)
        
        self.asignar_estudiantes_restantes()
        self.completar_equipos()
        
        return self.asignaciones, self.asignacion_estudiantes

In [52]:
def main(data):
    heuristica = GenerarEquipos(data)
    asignaciones, asignacion_estudiantes = heuristica.formar_equipos()
    
    if heuristica.validar_asignacion():
        print("Asignaciones válidas completadas")
        
        metricas = MetricasDistribucion(
            asignaciones=asignaciones,
            asignacion_estudiantes=asignacion_estudiantes,
            estudiantes=data['estudiantes'],  # Usar datos originales para métricas
            desafios=data['desafios']
        )
        
        metricas.entregar_metricas()
        heuristica.entregar_resultados_asignacion()
        
    else:
        print("Error en las asignaciones")
    
    return asignaciones, asignacion_estudiantes

In [57]:
# Cargar datos
import json
with open('body.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

# Ejecutar el algoritmo
asignaciones, asignacion_estudiantes = main(data)

Asignaciones válidas completadas

=== Métricas de Asignación ===
Satisfacción promedio: 0.745
Estudiantes en primera prioridad: 42
Estudiantes fuera de preferencias: 9
Desafíos sin equipo: 19
Desviación estándar tamaño equipos: 0.482
Promedio de carreras por equipo: 1.79

Porcentajes:
Primera prioridad: 65.6%
Segunda y Tercera prioridad: 20.3%
Fuera de preferencias: 14.1%

Estadísticas de asignación:

Desafío: Clasificación de imágenes de mamografía usando Machine Learning
Número de estudiantes: 4
Distribución por carrera:
- Ingeniería Civil Telemática: 3 (máximo permitido: 3)
- Ingeniería Civil Industrial: 1 (máximo permitido: 1)
Estudiantes en el equipo:
- Julio Aníbal Maturana Solís (Ingeniería Civil Telemática) (Preferencia #1)
- Carlos Alfredo Cea Rios (Ingeniería Civil Telemática) (Preferencia #1)
- Martín Alejandro Rojas Moya (Ingeniería Civil Telemática) (Preferencia #1)
- Battá Tomás Tuki Cadenas (Ingeniería Civil Industrial) (Preferencia #2)

Desafío: Uso de LLMs para masific